In [ ]:
# Importar librerías estándar para análisis numérico
import numpy as np

In [ ]:
# TensorFlow / Keras: API moderna para modelos secuenciales, capas y utilidades
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Input
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import StringLookup

In [ ]:
# Fijar semillas para reproducibilidad de resultados
np.random.seed(42)
tf.random.set_seed(42)

### Datos

In [ ]:
# Ruta al texto completo de Don Quijote de la Mancha
file_path = 'C:\\Users\\alvar\\Documents\\tensorflow_ejemplos\\DATA\\Quijote.txt'

In [ ]:
# Leer el archivo completo con codificación UTF-8
text = open(file_path, 'r', encoding='utf-8').read()

### Vectorización

In [ ]:
# Construir el vocabulario de caracteres únicos ordenados
char_vocabulary = sorted(set(text))
print(f'Caracteres únicos: {len(char_vocabulary)}')
print(char_vocabulary)

In [ ]:
# Capa para convertir caracteres a IDs enteros (sin token de máscara)
char_to_id = StringLookup(vocabulary=list(char_vocabulary), mask_token=None)

In [ ]:
# Capa inversa: convertir IDs de vuelta a caracteres
id_to_char = StringLookup(
    vocabulary=char_to_id.get_vocabulary(),
    invert=True,
    mask_token=None
)

In [ ]:
# Función auxiliar para reconstruir texto a partir de un tensor de IDs
def text_from_ids(ids):
    chars = id_to_char(ids)
    joined = tf.strings.reduce_join(chars, axis=-1)
    return joined.numpy().decode('utf-8')

In [ ]:
# Convertir todo el texto a una secuencia de IDs
text_as_ids = char_to_id(tf.strings.unicode_split(text, 'UTF-8'))

# Verificar la conversión mostrando los primeros 20 caracteres y sus IDs
print(f'Texto original: {text[:20]}')
print(f'IDs resultantes: {text_as_ids.numpy()[:20]}')

### Crear Secuencias y Batches

In [ ]:
# Longitud de cada secuencia de entrada (el target será la misma secuencia desplazada 1 paso)
sequence_length = 500

In [ ]:
# Dataset de TensorFlow que emite cada ID como un elemento individual
ids_dataset = tf.data.Dataset.from_tensor_slices(text_as_ids)

In [ ]:
# Agrupar IDs en secuencias contiguas de tamaño (sequence_length + 1)
# El +1 permite separar entrada y target dentro de cada secuencia
sequences_dataset = ids_dataset.batch(sequence_length + 1, drop_remainder=True)

In [ ]:
# Dividir cada secuencia en entrada (todos los caracteres menos el último)
# y target (todos los caracteres menos el primero)
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

In [ ]:
# Aplicar la división a todas las secuencias del dataset
dataset = sequences_dataset.map(split_input_target)

In [ ]:
# Mostrar un ejemplo de entrada y target para verificar el desplazamiento
for input_example, target_example in dataset.take(1):
    print('Secuencia de entrada:')
    print(f'  IDs: {input_example.numpy()}')
    print(f'  Texto: {text_from_ids(input_example)}\n')
    print('Secuencia target:')
    print(f'  IDs: {target_example.numpy()}')
    print(f'  Texto: {text_from_ids(target_example)}')

In [ ]:
# Hiperparámetros para el pipeline de datos
batch_size = 128
buffer_size = 10000

In [ ]:
# Configurar el dataset para entrenamiento: mezclar, agrupar en batches y prefetch
# tf.data.AUTOTUNE optimiza automáticamente el paralelismo de carga de datos
train_dataset = (
    dataset
    .shuffle(buffer_size)
    .batch(batch_size, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)
print(f'Dataset listo para entrenamiento: {train_dataset}')

### Crear el Modelo

In [ ]:
# Dimensiones del modelo
vocab_size = char_to_id.vocabulary_size()
embed_dim = 256
gru_units = 1024

In [ ]:
# Definir arquitectura del modelo usando la API Sequential moderna de Keras
# Input con shape=(None,) permite secuencias de cualquier longitud
def create_model(vocab_size, embed_dim, gru_units):
    model = Sequential([
        Input(shape=(None,)),                        # Entrada variable: (batch, longitud_secuencia)
        Embedding(input_dim=vocab_size, output_dim=embed_dim),  # Mapea cada ID a un vector denso
        GRU(gru_units, return_sequences=True),        # GRU moderna; devuelve una salida por timestep
        Dense(vocab_size)                             # Capa de salida: logits para cada carácter del vocabulario
    ])
    return model

In [ ]:
# Instanciar el modelo de entrenamiento
training_model = create_model(vocab_size, embed_dim, gru_units)

In [ ]:
# Mostrar resumen de la arquitectura y número de parámetros
training_model.summary()

In [ ]:
# Verificar las dimensiones de salida con un batch de ejemplo
# La salida debe tener forma (batch_size, sequence_length, vocab_size)
for input_example_batch, target_example_batch in train_dataset.take(1):
    example_batch_predictions = training_model(input_example_batch)
    print(f'Forma de la salida: {example_batch_predictions.shape}')

In [ ]:
# Compilar el modelo
# SparseCategoricalCrossentropy con from_logits=True evita una capa softmax extra
# y es numéricamente más estable
training_model.compile(
    optimizer='adam',
    loss=SparseCategoricalCrossentropy(from_logits=True)
)

### Entrenamiento

In [ ]:
# Número máximo de épocas; EarlyStopping detendrá el entrenamiento si la loss no mejora
epochs = 100

In [ ]:
# EarlyStopping: recupera los pesos de la mejor época si la loss se estanca
early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

In [ ]:
# Entrenar el modelo con el dataset preparado
history = training_model.fit(train_dataset, epochs=epochs, callbacks=[early_stop])

### Generar Nuevo Texto

In [ ]:
# Crear modelo de generación con GRU stateful para procesar 1 carácter a la vez
# stateful=True mantiene el estado oculto entre llamadas sucesivas
# Input(batch_size=1, shape=(None,)) permite cualquier longitud de secuencia inicial
generation_model = Sequential([
    Input(batch_size=1, shape=(None,)),
    Embedding(input_dim=vocab_size, output_dim=embed_dim),
    GRU(gru_units, return_sequences=True, stateful=True),
    Dense(vocab_size)
])

# Transferir los pesos entrenados al modelo de generación
generation_model.set_weights(training_model.get_weights())

In [ ]:
# Resumen del modelo de generación (misma arquitectura, pesos transferidos)
generation_model.summary()

In [ ]:
# Función para generar texto a partir de una semilla inicial
# temperature controla la aleatoriedad: valores bajos = texto más predecible
def generate_text(model, seed_text, generate_length=500, temperature=1.0):
    # Resetear estados internos de la GRU antes de cada generación
    for layer in model.layers:
        if hasattr(layer, 'reset_states'):
            layer.reset_states()

    # Convertir la semilla de texto a IDs
    seed_chars = tf.strings.unicode_split(seed_text, 'UTF-8')
    input_ids = char_to_id(seed_chars)
    input_ids = tf.expand_dims(input_ids, 0)  # Agregar dimensión de batch: (1, len_seed)

    # Procesar la semilla completa para inicializar el estado oculto de la GRU
    predictions = model(input_ids)  # (1, len_seed, vocab_size)

    generated_chars = []

    for _ in range(generate_length):
        # Usar solo la predicción del último timestep
        last_prediction = predictions[:, -1, :]  # (1, vocab_size)
        last_prediction = last_prediction / temperature

        # Muestrear el siguiente carácter usando distribución categórica
        predicted_id = tf.random.categorical(last_prediction, num_samples=1)[0, 0].numpy()

        generated_chars.append(id_to_char(predicted_id).numpy().decode('utf-8'))

        # Predecir el siguiente carácter a partir del último predicho (1 timestep a la vez)
        # La GRU mantiene el estado oculto gracias a stateful=True
        next_input = tf.expand_dims([predicted_id], 0)  # (1, 1)
        predictions = model(next_input)  # (1, 1, vocab_size)

    return seed_text + ''.join(generated_chars)

In [ ]:
# Generar texto con temperature=1.0 (máxima creatividad / aleatoriedad)
print(generate_text(generation_model, seed_text='En un lugar de La Mancha', generate_length=1000, temperature=1.0))

In [ ]:
# Generar texto con temperature=0.5 (más conservador y predecible)
print(generate_text(generation_model, seed_text='En un lugar de La Mancha', generate_length=1000, temperature=0.5))